# openAI Harmony

https://cookbook.openai.com/articles/openai-harmony

In [1]:
from openai_harmony import (
    Author,
    Conversation,
    DeveloperContent,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    StreamableParser,
    ToolDescription,
    load_harmony_encoding,
    ReasoningEffort
)
from mlx_lm import generate, load

In [2]:
checkpoint = "openai/gpt-oss-20b"
model, tokenizer = load(path_or_hf_repo=checkpoint)
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
stream = StreamableParser(encoding, role=Role.ASSISTANT)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [3]:
system_message = (
    SystemContent.new()
        .with_reasoning_effort(ReasoningEffort.HIGH)
        .with_conversation_start_date("2025-06-28")
)

In [4]:
developer_message = (
    DeveloperContent.new()
        .with_instructions("Always respond in riddles")
        .with_function_tools(
            [
                ToolDescription.new(
                    "get_current_weather",
                    "Gets the current weather in the provided location.",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "default": "celsius",
                            },
                        },
                        "required": ["location"],
                    },
                ),
            ]
	)
)

In [5]:
messages = [
        Message.from_role_and_content(Role.SYSTEM, system_message),
        Message.from_role_and_content(Role.DEVELOPER, developer_message),
        Message.from_role_and_content(Role.USER, "What is the weather in Tokyo?"),
        Message.from_role_and_content(
            Role.ASSISTANT,
            'User asks: "What is the weather in Tokyo?" We need to use get_current_weather tool.',
        ).with_channel("analysis"),
        Message.from_role_and_content(Role.ASSISTANT, '{"location": "Tokyo"}')
        .with_channel("commentary")
        .with_recipient("functions.get_current_weather")
        .with_content_type("<|constrain|> json"),
        Message.from_author_and_content(
            Author.new(Role.TOOL, "functions.get_current_weather"),
            '{ "temperature": 20, "sunny": true }',
        ).with_channel("commentary"),
    ]

In [6]:
messages

[Message(author=Author(role=<Role.SYSTEM: 'system'>, name=None), content=[SystemContent(model_identity='You are ChatGPT, a large language model trained by OpenAI.', reasoning_effort=<ReasoningEffort.HIGH: 'High'>, conversation_start_date='2025-06-28', knowledge_cutoff='2024-06', channel_config=ChannelConfig(valid_channels=['analysis', 'commentary', 'final'], channel_required=True), tools=None)], channel=None, recipient=None, content_type=None),
 Message(author=Author(role=<Role.DEVELOPER: 'developer'>, name=None), content=[DeveloperContent(instructions='Always respond in riddles', tools={'functions': ToolNamespaceConfig(name='functions', description=None, tools=[ToolDescription(name='get_current_weather', description='Gets the current weather in the provided location.', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'format': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'default': 'celsi

In [7]:
convo = Conversation.from_messages(messages)    

In [8]:
convo

Conversation(messages=[Message(author=Author(role=<Role.SYSTEM: 'system'>, name=None), content=[SystemContent(model_identity='You are ChatGPT, a large language model trained by OpenAI.', reasoning_effort=<ReasoningEffort.HIGH: 'High'>, conversation_start_date='2025-06-28', knowledge_cutoff='2024-06', channel_config=ChannelConfig(valid_channels=['analysis', 'commentary', 'final'], channel_required=True), tools=None)], channel=None, recipient=None, content_type=None), Message(author=Author(role=<Role.DEVELOPER: 'developer'>, name=None), content=[DeveloperContent(instructions='Always respond in riddles', tools={'functions': ToolNamespaceConfig(name='functions', description=None, tools=[ToolDescription(name='get_current_weather', description='Gets the current weather in the provided location.', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'format': {'type': 'string', 'enum': ['celsius', 'fahrenheit

In [9]:
tokens = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)

In [10]:
for token in tokens:
    print("--------------------------------")
    print(token)
    print("--------------------------------")
    try: 
        stream.process(token)
    except Exception as e:
        print(e)
    print("--------------------------------")
    print("current_role", stream.current_role)
    print("current_channel", stream.current_channel)
    print("last_content_delta", stream.last_content_delta)
    print("current_content_type", stream.current_content_type)
    print("current_recipient", stream.current_recipient)
    print("current_content", stream.current_content)
    print("--------------------------------")

--------------------------------
200006
--------------------------------
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta None
current_content_type None
current_recipient None
current_content 
--------------------------------
--------------------------------
17360
--------------------------------
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta None
current_content_type None
current_recipient None
current_content 
--------------------------------
--------------------------------
200008
--------------------------------
--------------------------------
current_role Role.ASSISTANT
current_channel None
last_content_delta None
current_content_type None
current_recipient <|start|>system
current_content 
--------------------------------
--------------------------------
3575
--------------------------------
--------------------------------
current_role Role.ASSISTANT
current_channel None
las

In [11]:
response = generate(model=model, tokenizer=tokenizer, prompt=tokens, max_tokens=1024, verbose=True)

<|channel|>final<|message|>In the city of crimson blossoms, the sky whispers a gentle 20 degrees, and the sun smiles upon the streets—no clouds to cloud its bright decree.
Prompt: 240 tokens, 355.555 tokens-per-sec
Generation: 36 tokens, 33.503 tokens-per-sec
Peak memory: 14.611 GB
